# 10 · spark-submit & Modos de Deploy (Caso C)

**Teoria**: docs/01-do-mapreduce-ao-spark.md ("Modo Client vs. Cluster")

**Pré-requisito**: `make up-hadoop` ainda em execução.

---

Todo notebook até agora usou **client mode** implicitamente — o Driver é
este próprio processo (ou, para os Casos B/D, o container `spark-connect` atuando
como um Driver permanente). Este laboratório torna a distinção de modo de deploy explícita
usando `spark-submit` diretamente de um terminal, de ambas as formas.

🎯 **Objetivo deste laboratório:**
1. Executar o mesmo Job em **client mode** — onde você vê os logs ao vivo
2. Executar o mesmo Job em **cluster mode** — onde o Driver roda dentro do YARN
3. Comparar as diferenças na prática e entender quando usar cada um

📌 **Conceito-chave**: a diferença entre client e cluster mode é **onde o processo Driver executa**:
- **Client mode**: Driver executa na máquina que chamou `spark-submit` (seu terminal)
- **Cluster mode**: Driver executa dentro de um container do YARN (em um NodeManager)

> ⚠️ **Nota importante**: este notebook não contém células Python executáveis. Os comandos
> abaixo devem ser executados em um **terminal separado** (não dentro do Jupyter).
> O `spark-submit` inicia sua própria JVM e bloqueia até o Job terminar.

### 🧠 Client vs Cluster: por que isso importa?

A escolha do modo de deploy afeta **três aspectos** críticos de uma aplicação Spark:

| Aspecto | Client Mode | Cluster Mode |
|---|---|---|
| **Onde o Driver roda** | Na sua máquina (ou no container cliente) | Dentro do cluster YARN (em um NodeManager) |
| **Visibilidade dos logs** | Logs aparecem no seu terminal em tempo real | Logs ficam no YARN — precisamos de `yarn logs` para ver |
| **Resiliência** | Seu laptop desconectar? O Job morre | O Job sobrevive independente do cliente |
| **Interatividade** | Perfeito para desenvolvimento iterativo (notebooks!) | Melhor para Jobs de produção agendados |
| **Latência de rede** | Driver perto dos dados (ideal para datasets grandes) | Driver perto dos dados (dentro do cluster) |

> 💡 **Client mode** é o que você usou em todos os notebooks até agora — o notebook
> Jupyter É o Driver. **Cluster mode** é o padrão para Jobs de produção que precisam
> executar mesmo se o desenvolvedor fechar o laptop.

## Client mode: o padrão que você vem usando em todo o laboratório

Execute isso em um **terminal** (não no notebook — `spark-submit` inicia sua
própria JVM e bloqueia até o Job terminar):

```bash
cat <<'EOF' > /tmp/word_count_job.py
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("word-count-client")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)
df = spark.range(0, 10_000_000)
print(f"Count: {df.count()}")
spark.stop()
EOF

uv run spark-submit --master yarn --deploy-mode client /tmp/word_count_job.py
```

Observe que os logs do driver são impressos diretamente no **seu terminal** — isso é client
mode: sua máquina É o Driver, acompanhando toda a execução ao vivo.

🎯 **O que observar no terminal:**
- As mensagens `INFO` do Spark mostram a conexão com o ResourceManager
- Você vê o ApplicationMaster sendo alocado
- O `print(f"Count: ...")` aparece diretamente na saída
- Se você interromper o comando (Ctrl+C), o Job é abortado — porque o Driver morre

### 💡 Anatomia de uma submissão em Client Mode

Quando você executa `spark-submit --deploy-mode client`, o seguinte acontece:

```
Seu terminal (Driver)
    │
    ├── 1. Conecta no ResourceManager (localhost:8088) e submete a aplicação
    │
    ├── 2. ResourceManager aloca um container para o ApplicationMaster
    │    └── ApplicationMaster negocia containers para os executores
    │
    ├── 3. Executores iniciam nos NodeManagers
    │    └── Estabelecem conexão de volta com o Driver em host.docker.internal
    │
    ├── 4. Driver envia as tasks para os executores
    │    └── Logs e resultados voltam para o terminal
    │
    └── 5. Job termina → Driver finaliza → containers são liberados
```

📌 **Dependência**: o `spark.driver.host=host.docker.internal` é crucial aqui
— os executores dentro dos containers Docker precisam de um endereço para
chamar o Driver de volta. Sem ele, a conexão nunca se estabelece.

> ⚠️ **Atenção**: ao contrário do notebook (que mantém a SparkSession viva para
> múltiplas células), o `spark-submit` executa o script inteiro e depois morre.
> Por isso que scripts standalone chamam `spark.stop()` no final.

## Cluster mode: driver executa dentro do próprio YARN

```bash
uv run spark-submit --master yarn --deploy-mode cluster /tmp/word_count_job.py
```

Desta vez seu terminal mostra apenas a **submissão** bem-sucedida — a
saída do `print()` *não* aparece localmente, porque o processo Driver está
agora executando dentro de um container YARN (agendado no nodemanager1 ou
nodemanager2), não na sua máquina.

Para ver sua saída, encontre o ID da Application impresso pelo spark-submit e
verifique seus logs na UI do ResourceManager (http://localhost:8088 → a aplicação → Logs), ou:

```bash
docker exec -it resourcemanager yarn logs -applicationId <application_id>
```

🎯 **O que observar:**
- O terminal volta rapidamente (só submete, não executa)
- `print()` não aparece localmente — o Driver está dentro do container
- A Application sobrevive mesmo se você fechar o terminal
- Para ver o resultado, precisa acessar os logs do YARN

> 💡 **Esta é a compensação do modo de deploy de docs/01, tornada concreta**:
> cluster mode significa que seu Job sobrevive ao seu laptop desconectar — porque
> seu laptop nunca estava executando o Driver em primeiro lugar. Client mode
> (que o truque compartilhado `spark.driver.host=host.docker.internal` em
> `lab_utils.py` torna possível a partir do seu HOST) troca essa resiliência pelo
> fluxo de trabalho interativo e ao vivo em torno do qual todo este laboratório é
> construído.

### 🧠 Anatomia de uma submissão em Cluster Mode

Em cluster mode, o fluxo é diferente:

```
Seu terminal (cliente leve)
    │
    ├── 1. Submete a aplicação ao ResourceManager
    │    └── spark-submit termina e exibe "application_id"
    │
    └── 2. ResourceManager aloca um container para o ApplicationMaster
         └── Neste modo, o ApplicationMaster É o Driver!
              │
              ├── Driver (dentro do container) negocia executores
              ├── Executores rodam tasks
              └── Resultados ficam nos logs do YARN
```

📌 **Diferença fundamental**:
- Client mode: Driver = spark-submit (2 processos separados: Driver + AM)
- Cluster mode: Driver = ApplicationMaster (1 processo que faz tudo)

> 💡 **Quando usar cada um?**
> - **Client mode**: desenvolvimento interativo, notebooks, debugging, quando você
>   quer ver os logs ao vivo
> - **Cluster mode**: Jobs de produção (ETL agendados), quando o cliente é instável,
>   quando você quer desacoplar a execução do terminal

⚠️ **Cluster mode com Docker**: neste laboratório, o cluster mode adicionalmente
não precisa de `spark.driver.host=host.docker.internal` porque o Driver está
dentro da rede Docker — ele NATURALMENTE alcança os executores.

## Observando na UI do ResourceManager

Para qualquer execução (client ou cluster mode), abra http://localhost:8088
enquanto o Job está em andamento:

1. Clique no ID da Application → veja o nó do Application Master e o
   número de containers (executores) alocados.
2. Observe **onde** o Application Master executa:
   - Em **client mode**: o AM é um coordenador leve; o Driver está no host
   - Em **cluster mode**: o AM **é** seu Driver
3. Compare o link "Tracking UI":
   - Client mode: Tracking UI aponta para a Spark UI no seu host
   - Cluster mode: Tracking UI aponta para a Spark UI dentro do container

📌 **Na prática, a Tracking UI é seu melhor diagnóstico**: se ela aponta para
um IP dentro da rede Docker (`172.x.x.x`), você está em cluster mode e não
consegue acessar diretamente do navegador — precisará usar `yarn logs`.

> 💡 **Resumo final**: o modo de deploy não muda a lógica do seu Job — apenas
> **onde** o Driver executa. O código PySpark é exatamente o mesmo nos dois modos.
> A diferença está em quem gerencia a vida do Driver e onde os logs aparecem.